# Colab Validation Run — Phases 1–6 on Real WOMD Data

Phases 1–6 are committed and tested, but **only against synthetic scenarios built as numpy
arrays** — two vehicles, all 91 timesteps valid, no crowding, no occlusion. Nothing in
Phases 4, 5, or 6 has ever touched a real WOMD shard. `export_shard_geometry` has never
executed at all.

This is a **validation run, not a demo**. The goal is diagnostics: find out where the
synthetic fixtures lied. Every cell below prints its findings even when the news is boring —
"0 interior gaps" is a result, not a skip.

Two design bugs are suspected going in, and this notebook is built to measure their blast
radius rather than assume it:

1. **Pass 1 scoring is SDC-agnostic.** `score_scenario` never receives `sdc_index`, so the
   fragility ranking may be measuring "how close did any two agents come" rather than
   "how close was the SDC to danger" — which is what Phase 4 actually optimizes.
2. **PET evaluates only one crossing order.** `compute_pet_pair` can return a spurious
   negative PET for a perfectly safe, sequenced crossing, which then scores as *maximally*
   dangerous. Cells 7c/7d measure both.

> ⚠️ **If you edit any file under `src/` and re-run a cell, Colab does NOT see the change.**
> Python caches the imported module. The fix is **Runtime → Restart session**, then re-run
> from the top — not `del sys.modules[...]`, which is fragile and easy to get subtly wrong
> with submodules.

This notebook writes only to a session-local Postgres database and reads the repo + the
shard. **It never writes back to the repo.**

## 1. Protobuf environment variable

Must be set **before any import**, in its own cell, before anything (including this
notebook's own later cells) imports `google.protobuf` transitively. Setting it after import
has no effect — the C++ implementation is already loaded.

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
print("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION =",
      os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'])

## 2. Installs

`--no-deps` on the Waymo package: it pins an old TensorFlow we do not need (`loader.py`
parses TFRecord with `struct`; only `scenario_pb2` from the Waymo package is used at
runtime). `torch` is deliberately **not installed** — `autograd_optimizer.py` imports it at
module level, but `batch_scorer._stress_one` only imports that module when
`use_autograd=True`, and this notebook runs DE-only, so it is never needed.

Every resolved version is printed so a future failure has a known-good baseline to diff
against.

In [ ]:
!pip install waymo-open-dataset-tf-2-11-0 --no-deps --quiet
!pip install shapely scipy psycopg2-binary --quiet

import importlib.metadata as ilm

for pkg in ["waymo-open-dataset-tf-2-11-0", "shapely", "scipy", "psycopg2-binary",
            "numpy", "protobuf"]:
    try:
        print(f"{pkg:35s} {ilm.version(pkg)}")
    except ilm.PackageNotFoundError:
        print(f"{pkg:35s} NOT INSTALLED")

import sys
print("\npython", sys.version)

## 3. Repo + Drive

Clones the repo if absent, otherwise pulls `main`. Mounts Drive and **asserts the shard file
exists before anything else runs** — failing here, loudly, beats discovering a missing file
300 lines into a batch pass.

`REPO_URL`, `SHARD_PATH` and the run-size knobs (`MAX_SCENARIOS`, `TOP_N`, `DIAG_N`) are the
only things you should need to change to rerun this at a different scale.

In [ ]:
# ── config (the only cell you should need to edit for a different run) ──────
REPO_URL = "https://github.com/AviShrivastava1/av_stress_tester.git"
REPO_DIR = "/content/av_stress_tester"
SHARD_PATH = ("/content/drive/MyDrive/waymo_data/"
              "uncompressed_scenario_training_training.tfrecord-00000-of-01000")

MAX_SCENARIOS = 100   # Pass 1 batch size
TOP_N = 5             # how many scenarios get the Phase 4 stress test (Pass 2)
DIAG_N = 50           # sample size for the 7c/7d rank-correlation + PET experiments
DE_KWARGS = dict(popsize=10, maxiter=60, tol=1e-2, seed=1)

PG_USER = "avi"
PG_DB = "av_stress"

In [ ]:
import os, subprocess, sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present — pulling latest main")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)
else:
    print(f"Cloning {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0] =", sys.path[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(SHARD_PATH), (
    f"Shard not found at {SHARD_PATH}. Check the Drive mount and the path before "
    f"running anything else — every later cell depends on this file."
)
size_gb = os.path.getsize(SHARD_PATH) / (1024 ** 3)
print(f"Shard found: {SHARD_PATH}")
print(f"Size: {size_gb:.2f} GB")

## 4. Postgres + PostGIS

Colab has no Postgres preinstalled. This installs it fresh, creates role `avi` and database
`av_stress`, enables PostGIS, and runs both schema-init functions. **Colab's disk resets
between sessions, so this database is per-session scratch — that is fine for a validation
run**; nothing here needs to persist.

`PGUSER`/`PGDATABASE`/`PGPASSWORD` are exported into the environment now because
`src/api/config.py` builds its `Settings()` singleton **at import time** — cell 14 must not
be the first thing to read these.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib postgis postgresql-14-postgis-3 > /dev/null
!service postgresql start

import subprocess as sp
sp.run(["sudo", "-u", "postgres", "psql", "-c",
        f"CREATE ROLE {PG_USER} WITH SUPERUSER LOGIN PASSWORD '';"],
       capture_output=True)
sp.run(["sudo", "-u", "postgres", "createdb", "-O", PG_USER, PG_DB],
       capture_output=True)
sp.run(["sudo", "-u", "postgres", "psql", "-d", PG_DB, "-c",
        "CREATE EXTENSION IF NOT EXISTS postgis;"], check=True)

os.environ['PGHOST'] = 'localhost'
os.environ['PGPORT'] = '5432'
os.environ['PGUSER'] = PG_USER
os.environ['PGPASSWORD'] = ''
os.environ['PGDATABASE'] = PG_DB

print(f"Postgres running, role={PG_USER}, database={PG_DB}")

In [ ]:
from src.scoring import db
from src.scoring.export_geometry import init_geometry_schema

conn = db.get_connection()
db.init_schema(conn)
init_geometry_schema(conn)

with conn.cursor() as cur:
    cur.execute("SELECT PostGIS_Version();")
    print("PostGIS_Version():", cur.fetchone()[0])
conn.commit()
print("Schema initialized: scenario_scores, scenario_agents, perturbed_paths")

## 5. Smoke test — one scenario

The cheapest possible check that Phase 1 still works against this specific shard, before
any batch work. If this cell fails, nothing downstream will work either.

In [ ]:
from src.data.loader import ShardLoader
from src.data.parser import ScenarioParser

loader = ShardLoader(SHARD_PATH)
raw = next(iter(loader))
parser = ScenarioParser(raw)

states = parser.get_agent_states()
validity = parser.get_agent_validity()
types = parser.get_agent_types()
sdc_idx = parser.get_sdc_index()

print("scenario_id:", parser.get_scenario_id())
print("states.shape:", states.shape)
print("validity.shape:", validity.shape)
print("sdc_index:", sdc_idx)

import numpy as np
uniq, counts = np.unique(types, return_counts=True)
type_names = {0: "UNSET", 1: "VEHICLE", 2: "PEDESTRIAN", 3: "CYCLIST", 4: "OTHER"}
print("agent type counts:")
for u, c in zip(uniq, counts):
    print(f"  {type_names.get(int(u), f'unknown({u})'):10s} {c}")

## 6. Timing probe, then Pass 1 (`score_shard`)

PET rebuilds swept-polygon unions inside `compute_pet_pair`, so its cost grows with agent
density — something the synthetic fixtures (2 agents) never exercised. This probes 3 real
scenarios first and **projects the cost for `MAX_SCENARIOS`**, so the run can be resized
before committing to it rather than discovered 20 minutes in.

In [ ]:
import time
from src.scoring.batch_scorer import _score_one

probe_records = []
t0 = time.time()
for i, raw in enumerate(ShardLoader(SHARD_PATH)):
    if i >= 3:
        break
    p = ScenarioParser(raw)
    rec = _score_one(p.get_agent_states(), p.get_agent_validity(), p.get_scenario_id())
    probe_records.append(rec)
probe_elapsed = time.time() - t0
per_scenario = probe_elapsed / len(probe_records)

print(f"Probe: {len(probe_records)} scenarios in {probe_elapsed:.2f}s "
      f"({per_scenario:.2f}s/scenario)")
print(f"Projected for MAX_SCENARIOS={MAX_SCENARIOS}: "
      f"{per_scenario * MAX_SCENARIOS:.1f}s (~{per_scenario * MAX_SCENARIOS / 60:.1f} min)")
print(f"Projected for a full 1000-shard dataset at this rate: "
      f"~{per_scenario * MAX_SCENARIOS * 1000 / 3600:.1f} hours "
      f"(scaling this probe's per-scenario cost, not a measured full run)")

In [ ]:
from src.scoring.batch_scorer import score_shard

t0 = time.time()
records, errors = score_shard(SHARD_PATH, max_scenarios=MAX_SCENARIOS,
                              pet_max_pairs=50, progress_every=25, verbose=True)
pass1_elapsed = time.time() - t0
print(f"\nPass 1 done: {len(records)} scored, {len(errors)} errored, "
      f"{pass1_elapsed:.1f}s total")

## 7. Pass 1 diagnostics

The most valuable cell in the batch section. Covers: error isolation (the first real
exercise of the per-scenario try/except in `score_shard`), the fragility distribution, TTC/PET
saturation rates, agent-count distribution, and validity-gap statistics across every parsed
track — including **interior gaps**, the case that breaks any "vertex index equals timestep"
assumption (see cell 13).

In [ ]:
import numpy as np
from collections import Counter

# ── error isolation ──────────────────────────────────────────────────────────
print("=" * 70)
print("ERROR ISOLATION")
print("=" * 70)
print(f"scored: {len(records)}   errored: {len(errors)}")
if errors:
    exc_types = Counter(e['error'].split(':')[0] for e in errors)
    print("distinct exception types:")
    for exc, n in exc_types.most_common():
        print(f"  {exc:30s} {n}")
else:
    print("0 errors — clean pass on this sample")

# ── fragility distribution ───────────────────────────────────────────────────
frag = np.array([r['fragility_score'] for r in records])
print()
print("=" * 70)
print("FRAGILITY SCORE DISTRIBUTION")
print("=" * 70)
print(f"min={frag.min():.4f}  max={frag.max():.4f}  mean={frag.mean():.4f}")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p:<3d} {np.percentile(frag, p):.4f}")

# ── TTC / PET saturation ──────────────────────────────────────────────────────
ttc = np.array([r['min_ttc'] for r in records])
pet = np.array([r['min_pet'] for r in records])

ttc_zero = (ttc == 0.0)
ttc_inf = (ttc == 999.0)
pet_zero = (pet == 0.0)
pet_inf = (pet == 999.0)

print()
print("=" * 70)
print("TTC / PET SATURATION (finding 1)")
print("=" * 70)
print(f"min_ttc == 0.0   (saturated, ALL pairs): {ttc_zero.sum()}/{len(ttc)} "
      f"({100*ttc_zero.mean():.1f}%)")
print(f"min_ttc == 999.0 (sentinel, no closing):  {ttc_inf.sum()}/{len(ttc)} "
      f"({100*ttc_inf.mean():.1f}%)")
print(f"min_pet == 0.0:                           {pet_zero.sum()}/{len(pet)} "
      f"({100*pet_zero.mean():.1f}%)")
print(f"min_pet == 999.0 (sentinel, never cross):  {pet_inf.sum()}/{len(pet)} "
      f"({100*pet_inf.mean():.1f}%)")

non_sat = frag[~ttc_zero]
print()
print(f"fragility percentiles over the NON-saturated subset (n={len(non_sat)}):")
if len(non_sat):
    for p in [10, 25, 50, 75, 90, 95, 99]:
        print(f"  p{p:<3d} {np.percentile(non_sat, p):.4f}")
else:
    print("  (every scenario in this sample saturated — see finding 1 in the summary cell)")

# ── agent count distribution ──────────────────────────────────────────────────
n_agents = np.array([r['n_agents'] for r in records])
print()
print("=" * 70)
print("AGENT COUNT (n_agents) DISTRIBUTION")
print("=" * 70)
print(f"min={n_agents.min()}  median={int(np.median(n_agents))}  max={n_agents.max()}")

# ── PET pair-selection diagnostics (finding 4) — index arithmetic only, no geometry ──
print()
print("=" * 70)
print("PET PAIR-SELECTION DIAGNOSTICS (finding 4)")
print("=" * 70)


def _pet_pairs_checked(n_agents_i, max_pairs=50):
    """Replicate compute_min_pet_scenario's (i, j) selection with i<j, no geometry."""
    pairs = []
    for i in range(n_agents_i):
        for j in range(i + 1, n_agents_i):
            if len(pairs) >= max_pairs:
                return pairs
            pairs.append((i, j))
    return pairs


ge_51 = 0
sdc_never_checked = 0
anchor_counts = []
for r in records:
    n = r['n_agents']
    if n >= 51:
        ge_51 += 1
    pairs = _pet_pairs_checked(n)
    anchors = len(set(i for i, j in pairs))
    anchor_counts.append(anchors)
    # sdc_idx is not stored on the record; re-derive is not available here without
    # re-parsing. This diagnostic instead reports the STRUCTURAL degeneracy (anchor
    # count) directly — see cell 7d for the SDC-specific version, computed on the
    # cached sample where sdc_idx is available.

print(f"scenarios with n_agents >= 51 (cap exhausted inside i=0): "
      f"{ge_51}/{len(records)} ({100*ge_51/len(records):.1f}%)")
print(f"distinct 'i' anchors reached per scenario: "
      f"min={min(anchor_counts)}  median={int(np.median(anchor_counts))}  "
      f"max={max(anchor_counts)}")
print("(anchors == 1 means every checked PET pair shares agent 0 in that scenario;")
print(" the SDC-specific version of this — whether the SDC is ever checked at all —")
print(" is computed in cell 7d against the cached sample, where sdc_idx is available.)")

# ── validity-gap statistics ───────────────────────────────────────────────────
print()
print("=" * 70)
print("VALIDITY-GAP STATISTICS")
print("=" * 70)
print("(recomputed per scenario in this pass — see cell 7b for the cached-sample version")
print(" used by cells 7c/7d; this section re-walks the shard once more, which is the")
print(" price of getting gap stats on the FULL MAX_SCENARIOS sample rather than DIAG_N)")

fully_valid = prefix_suffix_only = interior_gap = total_tracks = 0
t0 = time.time()
wanted_ids = set(r['scenario_id'] for r in records)
for raw in ShardLoader(SHARD_PATH):
    if not wanted_ids:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in wanted_ids:
        continue
    wanted_ids.discard(sid)
    v = p.get_agent_validity()
    for i in range(v.shape[0]):
        total_tracks += 1
        row = v[i]
        if row.all():
            fully_valid += 1
            continue
        valid_idx = np.where(row)[0]
        if len(valid_idx) == 0:
            continue
        # contiguous prefix/suffix only: the valid indices form one unbroken run
        span = valid_idx[-1] - valid_idx[0] + 1
        if span == len(valid_idx):
            prefix_suffix_only += 1
        else:
            interior_gap += 1
gap_scan_elapsed = time.time() - t0

print(f"total agent tracks examined: {total_tracks}  ({gap_scan_elapsed:.1f}s)")
print(f"  fully valid (all 91 timesteps):        {fully_valid} "
      f"({100*fully_valid/max(total_tracks,1):.1f}%)")
print(f"  contiguous run (no interior gap):       {prefix_suffix_only} "
      f"({100*prefix_suffix_only/max(total_tracks,1):.1f}%)")
print(f"  INTERIOR GAP (breaks index==timestep):   {interior_gap} "
      f"({100*interior_gap/max(total_tracks,1):.1f}%)")

# ── timing projection ──────────────────────────────────────────────────────
print()
print("=" * 70)
print("WALL-CLOCK PROJECTION")
print("=" * 70)
mean_sec = np.mean([r['score_seconds'] for r in records])
print(f"mean seconds/scenario (this pass): {mean_sec:.3f}s")
print(f"projected for a full 1000-shard dataset at ~{mean_sec:.3f}s/scenario "
      f"and this shard's scenario count as a stand-in: see the cell-6 probe for the "
      f"MAX_SCENARIOS-scale projection; this line uses the Pass-1-measured rate instead "
      f"of the 3-scenario probe rate, so compare the two for probe reliability.")

## 7b. Diagnostic sample cache

Cells 7c and 7d both need parsed arrays (`states`, `validity`, `types`, `sdc_idx`) for the
same sample of scenarios, and the shard lives on Drive-mounted storage where a second and
third sequential read would be the dominant cost of running this section. So this cell
parses `DIAG_N` scenarios once and holds them in memory.

Cheap to hold: a 100-agent scenario is `91 × 100 × 7 × 4 bytes ≈ 255 KB`, so `DIAG_N=50`
scenarios is on the order of tens of MB — printed below rather than assumed.

In [ ]:
import sys as _sys

diag_cache = []
for raw in ShardLoader(SHARD_PATH):
    if len(diag_cache) >= DIAG_N:
        break
    p = ScenarioParser(raw)
    diag_cache.append({
        'scenario_id': p.get_scenario_id(),
        'states': p.get_agent_states(),
        'validity': p.get_agent_validity(),
        'types': p.get_agent_types(),
        'sdc_idx': p.get_sdc_index(),
    })

resident_bytes = sum(
    d['states'].nbytes + d['validity'].nbytes + d['types'].nbytes
    for d in diag_cache
)
print(f"Cached {len(diag_cache)} scenarios for cells 7c/7d "
      f"({resident_bytes / 1024**2:.1f} MB resident)")

## 7c. Rank-correlation experiment — settles finding 1

Computes each cached scenario's danger profile **twice**, changing only the pair set:

- **all-pairs** — exactly what `score_scenario` does today
- **SDC-only** — the same underlying primitives (`compute_ttc_all_pairs`,
  `compute_pet_pair`), restricted to pairs involving the SDC

Both go through the real `compute_danger_score` and `rank_scenarios`, so only the *pair
selection* differs, not the scoring math. The PET pairs here keep the shipped `i < j`
orientation — the ordering bug is finding 5's subject and is measured separately in 7d, so
it does not confound this comparison.

**Interpretation:** high Spearman correlation and near-total top-N overlap means finding 1
is theoretical and Phase 3 stands as-is. A substantially different top-N list means the
cheap filter is measuring something different from what Phase 4 optimizes, and Phase 3
needs rework. This notebook diagnoses only — it does not change Phase 3.

Cost note: SDC-only PET is `N-1` **uncapped** pairs per scenario, so on dense scenarios this
cell can cost more than the whole Pass 1 batch. `DIAG_N` is tuned from the cell-6 probe.

In [ ]:
from src.danger.ttc_engine import compute_ttc_all_pairs, TTC_INFINITY
from src.danger.pet_engine import compute_pet_pair, PET_INFINITY
from src.danger.danger_score import compute_danger_score
from src.scoring.ranker import rank_scenarios

t0 = time.time()
all_pairs_records = []
sdc_only_records = []

for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']
    N, T, _ = states.shape

    # ---- all-pairs (current production behaviour) ----
    min_ttc_all = TTC_INFINITY
    for t in range(T):
        min_ttc_all = min(min_ttc_all, compute_ttc_all_pairs(states, validity, t).min())
    min_pet_all = PET_INFINITY
    checked = 0
    for i in range(N):
        for j in range(i + 1, N):
            if checked >= 50:
                break
            min_pet_all = min(min_pet_all, compute_pet_pair(states, validity, i, j))
            checked += 1
        if checked >= 50:
            break
    frag_all = compute_danger_score(min_ttc_all, min_pet_all)
    all_pairs_records.append({'scenario_id': d['scenario_id'], 'fragility_score': frag_all,
                              'min_ttc': min_ttc_all, 'min_pet': min_pet_all})

    # ---- SDC-only ----
    min_ttc_sdc = TTC_INFINITY
    for t in range(T):
        row = compute_ttc_all_pairs(states, validity, t)[sdc_idx]
        min_ttc_sdc = min(min_ttc_sdc, row.min())
    min_pet_sdc = PET_INFINITY
    for j in range(N):
        if j == sdc_idx:
            continue
        a, b = (sdc_idx, j) if sdc_idx < j else (j, sdc_idx)
        min_pet_sdc = min(min_pet_sdc, compute_pet_pair(states, validity, a, b))
    frag_sdc = compute_danger_score(min_ttc_sdc, min_pet_sdc)
    sdc_only_records.append({'scenario_id': d['scenario_id'], 'fragility_score': frag_sdc,
                             'min_ttc': min_ttc_sdc, 'min_pet': min_pet_sdc})

elapsed_7c = time.time() - t0
print(f"7c compute: {elapsed_7c:.1f}s for {len(diag_cache)} scenarios "
      f"({elapsed_7c/len(diag_cache):.2f}s/scenario)")

In [ ]:
from scipy.stats import spearmanr

ranked_all = rank_scenarios(all_pairs_records)
ranked_sdc = rank_scenarios(sdc_only_records)

order_all = [r['scenario_id'] for r in ranked_all]
order_sdc = [r['scenario_id'] for r in ranked_sdc]

rank_all = {sid: i for i, sid in enumerate(order_all)}
rank_sdc = {sid: i for i, sid in enumerate(order_sdc)}
common = list(rank_all.keys())

rho, pval = spearmanr([rank_all[s] for s in common], [rank_sdc[s] for s in common])

top10_all = set(order_all[:10])
top10_sdc = set(order_sdc[:10])
topN_all = set(order_all[:TOP_N])
topN_sdc = set(order_sdc[:TOP_N])

print("=" * 70)
print("RANK-CORRELATION EXPERIMENT (finding 1)")
print("=" * 70)
print(f"Spearman rho (all-pairs vs SDC-only ranking): {rho:.4f}  (p={pval:.4g})")
print(f"top-10 set overlap: {len(top10_all & top10_sdc)}/10")
print(f"top-{TOP_N} set overlap (the set Phase 4 would actually run on): "
      f"{len(topN_all & topN_sdc)}/{TOP_N}")

print()
print("scenarios whose rank moved most (all-pairs rank -> SDC-only rank):")
moves = sorted(common, key=lambda s: abs(rank_all[s] - rank_sdc[s]), reverse=True)[:10]
for sid in moves:
    print(f"  {sid}  all_rank={rank_all[sid]:4d} (frag={dict((r['scenario_id'], r) for r in ranked_all)[sid]['fragility_score']:.3f})"
          f"   sdc_rank={rank_sdc[sid]:4d} (frag={dict((r['scenario_id'], r) for r in ranked_sdc)[sid]['fragility_score']:.3f})")

if rho > 0.9 and len(topN_all & topN_sdc) == TOP_N:
    print("\nVERDICT: rankings closely agree — finding 1 appears theoretical on this sample.")
else:
    print("\nVERDICT: rankings diverge meaningfully — the cheap filter (Pass 1) is not")
    print("measuring what Phase 4 optimizes. This is evidence Phase 3 needs SDC-awareness.")

## 7d. PET both-orderings experiment — sizes finding 5

`compute_pet_pair(a, b)` computes only `t_exit_a` and `t_enter_b`, and returns
`enter_b - exit_a`. It never computes `enter_a` or `exit_b`. When agent `b` actually crosses
the conflict zone *before* agent `a`, the true PET is `enter_a - exit_b` (positive, safe),
but the code returns `enter_b - exit_a`, which is **negative** — indistinguishable from a
genuine simultaneous-occupancy collision.

This cell recomputes all four boundaries for **the pairs the shipped code actually
checks** (the capped, agent-0-anchored selection — not SDC pairs; the question here is what
today's `min_pet` is made of) and reports how many negative PETs are false alarms.

**Validity gate — non-negotiable.** The replicated `current` value must match
`compute_pet_pair` exactly on a sample, or this cell is measuring its own reimplementation
instead of the repo's behaviour. If the gate fails, every number below is suppressed.

In [ ]:
from src.danger.pet_engine import get_path_polygon, DT as PET_DT
from src.danger.collision_detector import get_corners
from shapely.geometry import Polygon


def _boundaries(states, validity, a, b):
    """
    Recompute all FOUR interval boundaries from one conflict-zone construction,
    mirroring pet_engine.compute_pet_pair's construction exactly so the validity
    gate below is a fair comparison.
    """
    path_a = get_path_polygon(states, a, validity)
    path_b = get_path_polygon(states, b, validity)
    if path_a is None or path_b is None:
        return None
    zone = path_a.intersection(path_b)
    if zone.is_empty:
        return None

    T = states.shape[1]

    def _enter_exit(agent):
        enter = exit_ = -1
        for t in range(T):
            if not validity[agent, t]:
                continue
            x, y = states[agent, t, 0], states[agent, t, 1]
            theta, length, width = states[agent, t, 4], states[agent, t, 5], states[agent, t, 6]
            box = Polygon(get_corners(x, y, theta, length, width))
            if box.intersects(zone):
                if enter == -1:
                    enter = t
                exit_ = t
        return enter, exit_

    enter_a, exit_a = _enter_exit(a)
    enter_b, exit_b = _enter_exit(b)
    if exit_a == -1 or enter_b == -1:
        return None
    return enter_a, exit_a, enter_b, exit_b


def _pet_pairs_checked(n_agents_i, max_pairs=50):
    pairs = []
    for i in range(n_agents_i):
        for j in range(i + 1, n_agents_i):
            if len(pairs) >= max_pairs:
                return pairs
            pairs.append((i, j))
    return pairs

In [ ]:
# ── validity gate: replicated 'current' must equal compute_pet_pair exactly ──
gate_pairs = []
for d in diag_cache:
    n = d['states'].shape[0]
    for (i, j) in _pet_pairs_checked(n):
        gate_pairs.append((d, i, j))
    if len(gate_pairs) >= 20:
        break

gate_ok = True
gate_checked = 0
for d, i, j in gate_pairs[:20]:
    b = _boundaries(d['states'], d['validity'], i, j)
    prod = compute_pet_pair(d['states'], d['validity'], i, j)
    if b is None:
        # both must agree there's no conflict zone / no valid crossing
        if prod not in (PET_INFINITY,):
            gate_ok = False
            print(f"GATE MISMATCH (no-zone case) sid={d['scenario_id']} pair=({i},{j}): "
                  f"replicated=None production={prod}")
        gate_checked += 1
        continue
    enter_a, exit_a, enter_b, exit_b = b
    current_replicated = (enter_b - exit_a) * PET_DT
    if abs(current_replicated - prod) > 1e-6:
        gate_ok = False
        print(f"GATE MISMATCH sid={d['scenario_id']} pair=({i},{j}): "
              f"replicated={current_replicated} production={prod}")
    gate_checked += 1

print(f"Validity gate: checked {gate_checked} pairs, "
      f"{'PASSED — replicated formula matches production exactly' if gate_ok else 'FAILED'}")
if not gate_ok:
    print("\n*** ABORTING 7d aggregates: the reimplementation above does not match")
    print("*** compute_pet_pair. Numbers below would measure the wrong thing.")

In [ ]:
if not gate_ok:
    print("(skipped — validity gate failed above)")
else:
    n_pairs = 0
    n_negative_current = 0
    n_flip_to_positive = 0
    n_stay_negative = 0
    per_scenario_min_current = {}
    per_scenario_min_corrected = {}

    for d in diag_cache:
        sid = d['scenario_id']
        n = d['states'].shape[0]
        min_current = PET_INFINITY
        min_corrected = PET_INFINITY
        for (i, j) in _pet_pairs_checked(n):
            b = _boundaries(d['states'], d['validity'], i, j)
            if b is None:
                continue
            enter_a, exit_a, enter_b, exit_b = b
            current = (enter_b - exit_a) * PET_DT
            corrected = max(enter_b - exit_a, enter_a - exit_b) * PET_DT
            n_pairs += 1
            if current < 0:
                n_negative_current += 1
                if corrected > 0:
                    n_flip_to_positive += 1
                else:
                    n_stay_negative += 1
            min_current = min(min_current, current)
            min_corrected = min(min_corrected, corrected)
        per_scenario_min_current[sid] = min_current
        per_scenario_min_corrected[sid] = min_corrected

    neg_scenarios = [s for s, v in per_scenario_min_current.items() if v < 0]
    flipped_scenarios = [s for s in neg_scenarios if per_scenario_min_corrected[s] > 0]

    print("=" * 70)
    print("PET BOTH-ORDERINGS EXPERIMENT (finding 5)")
    print("=" * 70)
    print(f"pairs evaluated: {n_pairs}")
    print(f"  negative under CURRENT formula:   {n_negative_current} "
          f"({100*n_negative_current/max(n_pairs,1):.1f}%)")
    print(f"    of those, FLIP to positive (false alarms): {n_flip_to_positive}")
    print(f"    of those, STAY negative (genuine overlap):  {n_stay_negative}")
    print()
    print(f"scenarios with min_pet < 0 (currently at pet_signal=100): "
          f"{len(neg_scenarios)}/{len(diag_cache)}")
    print(f"  of those, FLIP to positive once corrected (false alarms): "
          f"{len(flipped_scenarios)}/{max(len(neg_scenarios),1)}")
    print()
    print("HEADLINE: this many scenarios in the sample are sitting at maximum PET danger")
    print(f"that do NOT belong there: {len(flipped_scenarios)}")

In [ ]:
if not gate_ok:
    print("(skipped — validity gate failed)")
else:
    corrected_records = []
    for d in diag_cache:
        sid = d['scenario_id']
        min_pet_corrected = per_scenario_min_corrected[sid]
        # reuse the all-pairs TTC already computed in 7c for a fair fragility comparison
        matching = next(r for r in all_pairs_records if r['scenario_id'] == sid)
        frag = compute_danger_score(matching['min_ttc'], min_pet_corrected)
        corrected_records.append({'scenario_id': sid, 'fragility_score': frag})

    ranked_shipped = rank_scenarios(all_pairs_records)
    ranked_corrected = rank_scenarios(corrected_records)
    order_shipped = [r['scenario_id'] for r in ranked_shipped]
    order_corrected = [r['scenario_id'] for r in ranked_corrected]
    rank_shipped = {s: i for i, s in enumerate(order_shipped)}
    rank_corrected = {s: i for i, s in enumerate(order_corrected)}
    common = list(rank_shipped.keys())

    rho5, pval5 = spearmanr([rank_shipped[s] for s in common],
                            [rank_corrected[s] for s in common])
    topN_shipped = set(order_shipped[:TOP_N])
    topN_corrected = set(order_corrected[:TOP_N])

    print(f"Spearman rho (shipped PET vs corrected PET ranking): {rho5:.4f} (p={pval5:.4g})")
    print(f"top-{TOP_N} overlap: {len(topN_shipped & topN_corrected)}/{TOP_N}")

## 8. Rank + persist

Ranks the real Pass 1 output, prints the top 10, and persists via `upsert_scores`. Then
**upserts the same records a second time** and asserts the row count is unchanged —
idempotency verified against real data, not three synthetic rows.

In [ ]:
ranked = rank_scenarios(records)
print(f"{'rank':<6}{'scenario_id':<40}{'fragility':<12}{'min_ttc':<10}{'min_pet':<10}{'n_agents'}")
for r in ranked[:10]:
    print(f"{r['rank']:<6}{r['scenario_id']:<40}{r['fragility_score']:<12.4f}"
          f"{r['min_ttc']:<10.3f}{r['min_pet']:<10.3f}{r['n_agents']}")

In [ ]:
n1 = db.upsert_scores(conn, records)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM scenario_scores")
    count_after_first = cur.fetchone()[0]

n2 = db.upsert_scores(conn, records)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM scenario_scores")
    count_after_second = cur.fetchone()[0]

print(f"first upsert:  wrote {n1} rows, table now has {count_after_first}")
print(f"second upsert: wrote {n2} rows, table now has {count_after_second}")
assert count_after_first == count_after_second, (
    "IDEMPOTENCY VIOLATION: row count changed on a repeat upsert of identical records."
)
print("Idempotency confirmed on real data.")

## 9. Pass 2 — `stress_test_scenarios`

Runs the Phase 4 optimizer on the top `TOP_N` scenarios by fragility, with a small DE budget
so this stays fast enough to iterate on. Timed for the Pass-2 cost projection.

In [ ]:
from src.scoring.ranker import top_n_ids
from src.scoring.batch_scorer import stress_test_scenarios

ids_to_test = top_n_ids(records, TOP_N)
print("stress-testing:", ids_to_test)

t0 = time.time()
stress_results = stress_test_scenarios(SHARD_PATH, ids_to_test, de_kwargs=DE_KWARGS,
                                       verbose=True)
pass2_elapsed = time.time() - t0
print(f"\nPass 2 done: {pass2_elapsed:.1f}s for {len(ids_to_test)} scenarios "
      f"({pass2_elapsed/len(ids_to_test):.1f}s/scenario)")

## 10. Pass 2 diagnostics

Per-scenario table plus aggregates: how many hit `no_challenger`/`error`, how many
challengers were non-vehicles (the first real exercise of the linear-model path and of the
DE-only branch, since autograd is vehicle-only), and which perturbation component dominated
each solution — measured in `space.weights`-normalized units so speed (m/s) and heading
(rad) are comparable.

In [ ]:
type_names = {0: "UNSET", 1: "VEHICLE", 2: "PEDESTRIAN", 3: "CYCLIST", 4: "OTHER"}
component_names = ["dv0/dvx0", "dtheta0/dvy0", "da_bias/dax_bias", "ddelta_bias/day_bias"]

# re-parse just these scenarios to get types + build a PerturbationSpace for the weights
from src.optimization.perturbation_space import PerturbationSpace

challenger_types = {}
dominant_component = {}
for raw in ShardLoader(SHARD_PATH):
    if not ids_to_test:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in stress_results:
        continue
    r = stress_results[sid]
    if r.get('status') != 'ok' or r.get('target_idx') is None:
        continue
    types_arr = p.get_agent_types()
    tgt = r['target_idx']
    challenger_types[sid] = int(types_arr[tgt])

    space = PerturbationSpace(p.get_agent_states(), p.get_agent_validity(), types_arr,
                              p.get_sdc_index(), tgt)
    weighted = np.abs(np.asarray(r['delta']) * space.weights)
    dominant_component[sid] = component_names[int(np.argmax(weighted))]

print(f"{'scenario_id':<40}{'status':<14}{'collision':<11}{'||delta||':<12}"
      f"{'t_hit':<8}{'target_idx':<12}{'challenger_type':<16}{'dominant'}")
n_no_challenger = n_error = n_collision = n_safe = n_non_vehicle = 0
for sid, r in stress_results.items():
    status = r.get('status')
    if status == 'no_challenger':
        n_no_challenger += 1
    elif status == 'error':
        n_error += 1
    collided = r.get('collision', False)
    if status == 'ok':
        if collided:
            n_collision += 1
        else:
            n_safe += 1
    ttype = challenger_types.get(sid)
    ttype_name = type_names.get(ttype, "?") if ttype is not None else "-"
    if ttype is not None and ttype != 1:
        n_non_vehicle += 1
    print(f"{sid:<40}{status:<14}{str(collided):<11}"
          f"{r.get('min_perturbation', float('nan')):<12.4f}"
          f"{r.get('collision_timestep', -1):<8}{r.get('target_idx', -1):<12}"
          f"{ttype_name:<16}{dominant_component.get(sid, '-')}")

print()
print("=" * 70)
print("PASS 2 AGGREGATES")
print("=" * 70)
print(f"no_challenger: {n_no_challenger}   error: {n_error}")
print(f"collision found: {n_collision}   robustly safe: {n_safe}")
print(f"non-vehicle challengers (types 0/2/3/4, first real linear-model exercise): "
      f"{n_non_vehicle}/{len(challenger_types)}")
type4_or_0 = sum(1 for t in challenger_types.values() if t in (0, 4))
print(f"  of which TYPE_UNSET(0) or TYPE_OTHER(4) specifically: {type4_or_0}")
print(f"seconds/scenario (from cell 9): {pass2_elapsed/max(len(ids_to_test),1):.1f}")
print(f"implied cost of a top-50 stress test: "
      f"{pass2_elapsed/max(len(ids_to_test),1)*50/60:.1f} min")

## 11. `update_stress_results`, then re-read

Writes the Phase 4 columns, then re-reads via `db.fetch_top` and confirms the new columns
landed and the Pass-1 columns (`fragility_score`, `min_ttc`, `min_pet`) were **untouched**.

In [ ]:
n_updated = db.update_stress_results(conn, stress_results)
print(f"updated {n_updated} rows")

top_rows = db.fetch_top(conn, n=TOP_N)
for row in top_rows:
    print(f"{row['scenario_id']:<40} stress_tested_at={row['stress_tested_at']} "
          f"min_perturbation={row['min_perturbation']} "
          f"fragility_score={row['fragility_score']:.4f}")

for row in top_rows:
    orig = next(r for r in records if r['scenario_id'] == row['scenario_id'])
    assert abs(row['fragility_score'] - orig['fragility_score']) < 1e-9, (
        f"Pass-1 fragility_score for {row['scenario_id']} changed after a Pass-2 write "
        f"— update_stress_results must not touch Pass-1 columns."
    )
print("\nConfirmed: Phase 4 columns landed, Pass-1 columns untouched.")

## 12. Pass 3 — `export_shard_geometry`

**This code has never executed before this cell, on any data, synthetic or real.** It
shipped verified only by import. Prints the full summary dict and every error verbatim.

In [ ]:
from src.scoring.export_geometry import export_shard_geometry

t0 = time.time()
summary = export_shard_geometry(conn, SHARD_PATH, ids_to_test,
                                stress_results=stress_results, verbose=True)
pass3_elapsed = time.time() - t0

print()
print("=" * 70)
print("PASS 3 SUMMARY (export_shard_geometry — first-ever execution)")
print("=" * 70)
print(f"exported:           {summary['exported']}")
print(f"agents_written:     {summary['agents_written']}")
print(f"agents_skipped:     {summary['agents_skipped']}")
print(f"perturbed_written:  {summary['perturbed_written']}")
print(f"errors:             {len(summary['errors'])}")
for e in summary['errors']:
    print(f"  {e}")
print(f"elapsed: {pass3_elapsed:.1f}s")

## 13. Geometry verification — the most important assertion in this notebook

Re-parses the tested scenarios directly and compares the database against ground truth
computed independently in Python.

**The M-array comparison is the key assertion.** With every timestep valid (as in every
synthetic fixture used until now), `ST_M` values equal `0..N-1`, indistinguishable from
plain vertex indices — which is exactly why the original (buggy) exporter design could pass
every prior test. On real data with occlusion, an agent's M sequence should **jump** at a
gap. This cell proves the exporter wrote true timesteps, not vertex positions, by comparing
element-for-element against `np.where(validity[i])[0]`.

In [ ]:
def dump_points_m(conn, table, scenario_id, agent_idx=None):
    with conn.cursor() as cur:
        if table == 'scenario_agents':
            cur.execute("""
                SELECT agent_idx, ST_NPoints(path), headings,
                       ARRAY(SELECT ST_M(dp.geom) FROM ST_DumpPoints(path) dp
                             ORDER BY dp.path)
                FROM scenario_agents WHERE scenario_id = %s AND agent_idx = %s
            """, (scenario_id, agent_idx))
        else:
            cur.execute("""
                SELECT target_idx, ST_NPoints(path), headings,
                       ARRAY(SELECT ST_M(dp.geom) FROM ST_DumpPoints(path) dp
                             ORDER BY dp.path)
                FROM perturbed_paths WHERE scenario_id = %s
            """, (scenario_id,))
        return cur.fetchone()

In [ ]:
n_checked = 0
n_exact_match = 0
gap_example_found = False
skip_reconciled = 0

for raw in ShardLoader(SHARD_PATH):
    if not ids_to_test:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in ids_to_test:
        continue
    validity = p.get_agent_validity()
    N = validity.shape[0]

    with conn.cursor() as cur:
        cur.execute("SELECT agent_idx FROM scenario_agents WHERE scenario_id = %s", (sid,))
        exported_idxs = set(row[0] for row in cur.fetchall())

    for i in range(N):
        true_valid_ts = np.where(validity[i])[0]
        if len(true_valid_ts) < 2:
            skip_reconciled += int(i not in exported_idxs)
            continue
        if i not in exported_idxs:
            print(f"MISMATCH: agent {i} in {sid} has {len(true_valid_ts)} valid "
                  f"timesteps but was not exported")
            continue

        row = dump_points_m(conn, 'scenario_agents', sid, i)
        _, npoints, headings, m_values = row
        n_checked += 1

        assert npoints == len(true_valid_ts), (
            f"{sid} agent {i}: ST_NPoints={npoints} != {len(true_valid_ts)} valid timesteps"
        )
        m_array = np.array(m_values, dtype=int)
        if np.array_equal(m_array, true_valid_ts):
            n_exact_match += 1
        else:
            print(f"MISMATCH: {sid} agent {i}: M values {m_array[:10]}... != "
                  f"true valid timesteps {true_valid_ts[:10]}...")

        assert len(headings) == npoints, (
            f"{sid} agent {i}: len(headings)={len(headings)} != ST_NPoints={npoints}"
        )

        # is this a genuine interior-gap agent? report the first one found, verbatim.
        if not gap_example_found and len(true_valid_ts) >= 2:
            span = true_valid_ts[-1] - true_valid_ts[0] + 1
            if span != len(true_valid_ts):
                jump_pos = np.where(np.diff(true_valid_ts) > 1)[0][0]
                print(f"\nINTERIOR GAP EXAMPLE — {sid} agent {i}:")
                print(f"  M sequence around the jump: "
                      f"...{m_array[max(0,jump_pos-2):jump_pos+3]}...")
                print(f"  (jumps from {m_array[jump_pos]} to {m_array[jump_pos+1]}, "
                      f"skipping {m_array[jump_pos+1]-m_array[jump_pos]-1} invalid timesteps)")
                gap_example_found = True

print(f"\nagents checked: {n_checked}   exact M-array matches: {n_exact_match}")
assert n_exact_match == n_checked, "Some agents' M arrays did not match true valid timesteps."
print("CONFIRMED: M ordinate equals true valid-timestep indices, element-for-element.")

if not gap_example_found:
    print("\nNo interior-gap agent found among the tested scenarios' agents — reported")
    print("explicitly rather than silently: this sample happened not to contain one.")

print(f"\nagents with <2 valid timesteps, correctly skipped (not crashed on): "
      f"{skip_reconciled}")
print(f"reconciles against Pass 3's agents_skipped={summary['agents_skipped']} "
      f"(this loop only covers the {len(ids_to_test)} Pass-3 scenarios, so equality is "
      f"expected only if agents_skipped was computed over exactly this set)")

## 14. API check — the full round trip on real data

Starts uvicorn in a daemon thread (modern uvicorn skips installing signal handlers off the
main thread, so `Server.run()` works here) and polls `/health` until ready. `PGUSER` /
`PGPASSWORD` / `PGDATABASE` were exported back in cell 4 — before this is the first import
of `src.api.main`, since `Settings()` builds itself at import time.

The closing assertion is the point of this cell: confirm the `timesteps` array the HTTP
endpoint returns for a real scenario matches the M values read directly from Postgres in
cell 13 — proving the chain shard → exporter → PostGIS → HTTP holds together on real data.

In [ ]:
import threading
import uvicorn
import httpx

from src.api.main import app

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

base = "http://127.0.0.1:8000"
deadline = time.time() + 30
ready = False
last_error = None
while time.time() < deadline:
    try:
        r = httpx.get(f"{base}/health", timeout=1.0)
        if r.status_code == 200:
            ready = True
            break
        last_error = f"HTTP {r.status_code}: {r.text[:200]}"
    except httpx.HTTPError as e:
        # Expected while uvicorn is still binding its socket — connection refused
        # is the normal state for the first second or so. Not silently swallowed:
        # the last one seen is reported below if the deadline expires without success.
        last_error = f"{type(e).__name__}: {e}"
    time.sleep(0.5)

assert ready, (
    f"uvicorn did not become ready within 30s. Last error: {last_error}. "
    f"Check PGUSER/PGPASSWORD/PGDATABASE were exported in cell 4 before this import."
)
print("Server ready.")

In [ ]:
print("GET /health"); print(httpx.get(f"{base}/health").json())
print()
print("GET /stats"); print(httpx.get(f"{base}/stats").json())
print()
print("GET /scenarios?limit=3")
print(httpx.get(f"{base}/scenarios", params={"limit": 3}).json())

stress_tested_sid = next(sid for sid, r in stress_results.items()
                         if r.get('status') == 'ok' and r.get('collision'))
print(f"\nGET /scenarios/{stress_tested_sid}/trajectories")
traj = httpx.get(f"{base}/scenarios/{stress_tested_sid}/trajectories").json()
print(f"  {len(traj['agents'])} agents; first agent timesteps[:10] = "
      f"{traj['agents'][0]['timesteps'][:10]}")

print(f"\nGET /scenarios/{stress_tested_sid}/perturbed")
pert = httpx.get(f"{base}/scenarios/{stress_tested_sid}/perturbed").json()
print(f"  delta={pert['delta']}  collision_timestep={pert['collision_timestep']}")

In [ ]:
# close the loop: HTTP timesteps must equal the M values read directly from Postgres
target_idx_api = pert['target_idx']
http_agent = next(a for a in traj['agents'] if a['agent_idx'] == target_idx_api)
http_timesteps = np.array(http_agent['timesteps'])

row = dump_points_m(conn, 'scenario_agents', stress_tested_sid, target_idx_api)
_, npoints_db, _, m_values_db = row
db_m = np.array(m_values_db, dtype=float)

assert np.array_equal(http_timesteps, db_m), (
    "HTTP /trajectories timesteps do not match the M values read directly from Postgres — "
    "the shard-to-HTTP round trip is broken somewhere between cells 12 and 14."
)
print("CONFIRMED: HTTP timesteps == Postgres M values. Full round trip closed on real data.")

server.should_exit = True

## 15. Summary

Self-contained — safe to copy out of the notebook on its own. Leads with the two numbers
that decide whether Phase 3 needs rework.

In [ ]:
print("=" * 70)
print("COLAB VALIDATION RUN — SUMMARY")
print("=" * 70)
print()
print("-- Findings that decide whether Phase 3 needs rework --")
print(f"[Finding 1] all-pairs vs SDC-only ranking:  Spearman rho={rho:.4f}, "
      f"top-{TOP_N} overlap={len(topN_all & topN_sdc)}/{TOP_N}")
if gate_ok:
    print(f"[Finding 5] scenarios with spurious-negative min_pet (false alarms): "
          f"{len(flipped_scenarios)}/{len(diag_cache)}")
else:
    print(f"[Finding 5] validity gate FAILED — see cell 7d, numbers not computed")
print()
print("-- Pass 1 --")
print(f"scenarios scored: {len(records)}   errored: {len(errors)}")
print(f"fragility range: [{frag.min():.4f}, {frag.max():.4f}]  mean={frag.mean():.4f}")
print(f"TTC saturation (min_ttc==0.0, all-pairs): {100*ttc_zero.mean():.1f}%")
print(f"agent tracks with an interior validity gap: {interior_gap}/{total_tracks}")
print()
print("-- Pass 2 --")
print(f"stress-tested: {len(ids_to_test)}   collisions found: {n_collision}   "
      f"robustly safe: {n_safe}")
print(f"non-vehicle challengers: {n_non_vehicle}/{len(challenger_types)}")
print()
print("-- Pass 3 (export_shard_geometry — first execution ever) --")
print(f"exported: {summary['exported']}   agents_written: {summary['agents_written']}   "
      f"agents_skipped: {summary['agents_skipped']}   errors: {len(summary['errors'])}")
print(f"geometry M-ordinate verification: "
      f"{'PASSED' if n_exact_match == n_checked else 'FAILED'} "
      f"({n_exact_match}/{n_checked} agents)")
print()
print("-- Wall clock --")
print(f"Pass 1: {pass1_elapsed:.1f}s   Pass 2: {pass2_elapsed:.1f}s   "
      f"Pass 3: {pass3_elapsed:.1f}s")
print(f"total notebook compute (excludes installs/mount): "
      f"{pass1_elapsed + pass2_elapsed + pass3_elapsed:.1f}s")